# Epistemic Relevance Abstraction for Multi-Agent Coordination
## A Pedagogical Companion Notebook

**Paper:** *Epistemic Relevance Abstraction for Multi-Agent Coordination* — Sandy Tanwisuth (2025)

---

### What this notebook covers

This notebook walks through the key ideas in the paper using:

- **Plain-language intuitions** before every formal result
- **Toy matrix games** you can run and modify
- **Visual diagrams** of the full pipeline
- **Working code** for every major component

No game theory or multi-agent RL background is assumed — we build everything from scratch.

### The one-sentence version

> When you're coordinating with a partner, many surface-level differences in their behavior don't actually change what *you* should do. This paper gives you a way to figure out which differences matter, learn a compressed representation that keeps only those differences, and certify that acting on this compression is safe.

---
## 0. Setup & Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from scipy.special import softmax, rel_entr
from scipy.spatial.distance import cosine
from itertools import product

np.random.seed(42)

# Plotting style
plt.rcParams.update({
    'figure.facecolor': '#fafafa',
    'axes.facecolor': '#fafafa',
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

# Color palette
C = {
    'blue': '#3b82f6', 'red': '#ef4444', 'green': '#22c55e',
    'purple': '#a855f7', 'orange': '#f97316', 'gray': '#6b7280',
    'dark': '#1e293b', 'light': '#f1f5f9'
}

---
## 1. The Core Question: Which Differences Matter?

### Analogy: The barista problem

Imagine you're a barista (the **ego agent**). Different customers (the **co-agents**) walk in with different behaviors:

- Customer A orders a latte, sits quietly, reads a book.
- Customer B orders a latte, talks loudly on the phone, paces around.
- Customer C orders a pour-over, asks for specific grind size, watches you work.

From *your* perspective as the barista:
- A and B are **strategically equivalent**: despite very different *behaviors*, your best response is the same — make a latte.
- C is **strategically different**: you need to change what you do.

The paper asks: **can we automatically discover these groupings, learn them from data, and certify that acting on them is safe?**

### The pipeline at a glance

```
Co-policies  ──▶  Soft Best Responses  ──▶  Strategic InfoNCE  ──▶  SEC Embeddings  ──▶  ΔSEC Certificate
(what partners do)  (what ego should do)    (contrastive learning)  (compressed groups)   (is it safe to act?)
```

Let's build each piece.

---
## 2. A Toy World: The Coordination Matrix Game

We'll use a simple 2-player, 3-action matrix game throughout this notebook.

The **ego agent** (row player) and a **partner** (column player) each pick an action simultaneously. The ego gets a reward based on the joint outcome.

Think of it as: the ego needs to figure out what kind of partner it's dealing with and respond accordingly.

In [ ]:
# Ego agent's payoff matrix: R[ego_action, partner_action]
# Rows = ego's actions (0, 1, 2), Columns = partner's actions (0, 1, 2)
R = np.array([
    [10,  0,  3],   # Ego action 0
    [ 0,  8,  0],   # Ego action 1
    [ 3,  0,  7],   # Ego action 2
])

action_names = ['Rock', 'Paper', 'Scissors']  # just labels for intuition

print("Ego's payoff matrix R[ego_action, partner_action]:")
print("="*45)
header = f"{'':>12}" + "".join(f"{action_names[j]:>10}" for j in range(3))
print(header)
print("-"*45)
for i in range(3):
    row = f"{action_names[i]:>12}" + "".join(f"{R[i,j]:>10}" for j in range(3))
    print(row)

print("\nReading: If ego plays Rock and partner plays Rock, ego gets 10.")
print("Key structure: ego wants to 'match' certain partner actions.")

### Defining partner policies

A partner **policy** is a probability distribution over their actions. Let's define several partners that behave differently on the surface but may or may not require different responses from the ego.

In [ ]:
# Partner policies: probability distributions over {Rock, Paper, Scissors}
partners = {
    'Alice':   np.array([0.9, 0.05, 0.05]),  # Mostly plays Rock
    'Bob':     np.array([0.8, 0.1, 0.1]),     # Also mostly plays Rock (slightly noisier)
    'Carol':   np.array([0.7, 0.15, 0.15]),   # Still Rock-heavy but less extreme
    'Dave':    np.array([0.05, 0.9, 0.05]),   # Mostly plays Paper
    'Eve':     np.array([0.1, 0.1, 0.8]),     # Mostly plays Scissors
    'Frank':   np.array([0.33, 0.34, 0.33]),  # Uniform (unpredictable)
}

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(3)
width = 0.12
colors = [C['blue'], C['blue'], C['blue'], C['red'], C['green'], C['gray']]

for idx, (name, policy) in enumerate(partners.items()):
    bars = ax.bar(x + idx * width, policy, width, label=name,
                  color=colors[idx], alpha=0.8, edgecolor='white', linewidth=0.5)

ax.set_xticks(x + 2.5 * width)
ax.set_xticklabels(action_names)
ax.set_ylabel('Probability')
ax.set_title('Partner Policies: Who Does What?', fontweight='bold')
ax.legend(loc='upper right', framealpha=0.9)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

print("Notice: Alice, Bob, and Carol all favor Rock but at different intensities.")
print("The question is: does the ego need to RESPOND differently to each of them?")

---
## 3. Soft Best Responses: The Epistemic Primitive

### What's a best response?

Given a partner's policy, the ego's **expected payoff** for each action is:

$$Q(a_\text{ego}) = \sum_{a_\text{partner}} R[a_\text{ego}, a_\text{partner}] \cdot \pi_\text{partner}(a_\text{partner})$$

The **hard best response** picks the action with the highest Q-value (argmax). But argmax is brittle — a tiny payoff change can flip the answer.

The **soft best response** uses a Boltzmann (softmax) distribution instead:

$$\text{BR}^\tau(a \mid \pi_{-i}) = \frac{\exp(Q(a) / \tau)}{\sum_{a'} \exp(Q(a') / \tau)}$$

- $\tau \to 0$: recovers the hard best response (all mass on the best action)
- $\tau \to \infty$: uniform distribution (ignores payoffs entirely)
- Moderate $\tau$: a smooth, differentiable compromise

In [ ]:
def compute_q_values(R, partner_policy):
    """Expected payoff for each ego action given a partner policy."""
    return R @ partner_policy

def soft_best_response(R, partner_policy, tau=1.0):
    """Boltzmann soft best response at temperature tau."""
    q = compute_q_values(R, partner_policy)
    return softmax(q / tau)

# Show Q-values and soft BRs for each partner
tau = 0.5
print(f"Temperature τ = {tau}")
print(f"{'Partner':<10} {'Q-values':>25}   {'Soft BR':>25}   {'Hard BR':>12}")
print("="*80)

for name, policy in partners.items():
    q = compute_q_values(R, policy)
    sbr = soft_best_response(R, policy, tau=tau)
    hard_br = action_names[np.argmax(q)]
    q_str = ", ".join(f"{v:.2f}" for v in q)
    sbr_str = ", ".join(f"{v:.3f}" for v in sbr)
    print(f"{name:<10} [{q_str:>22}]   [{sbr_str:>22}]   {hard_br:>12}")

In [ ]:
# Visualize how temperature affects the soft BR for one partner
taus = np.logspace(-1.5, 1.5, 100)
partner_policy = partners['Alice']

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: soft BR vs tau for Alice
ax = axes[0]
sbrs = np.array([soft_best_response(R, partner_policy, tau=t) for t in taus])
for a_idx, a_name in enumerate(action_names):
    ax.plot(taus, sbrs[:, a_idx], linewidth=2.5,
            label=f'{a_name}', color=[C['blue'], C['red'], C['green']][a_idx])
ax.set_xscale('log')
ax.set_xlabel('Temperature τ (log scale)')
ax.set_ylabel('BR probability')
ax.set_title("Soft BR for Alice (mostly Rock)", fontweight='bold')
ax.legend()
ax.axvline(x=tau, color=C['gray'], linestyle='--', alpha=0.5, label=f'τ={tau}')

# Right: soft BRs at tau=0.5 for ALL partners
ax = axes[1]
partner_names = list(partners.keys())
sbr_matrix = np.array([soft_best_response(R, p, tau=tau) for p in partners.values()])

x_pos = np.arange(len(partner_names))
bottom = np.zeros(len(partner_names))
for a_idx, a_name in enumerate(action_names):
    color = [C['blue'], C['red'], C['green']][a_idx]
    ax.bar(x_pos, sbr_matrix[:, a_idx], bottom=bottom, label=a_name,
           color=color, alpha=0.85, edgecolor='white', linewidth=0.5)
    bottom += sbr_matrix[:, a_idx]

ax.set_xticks(x_pos)
ax.set_xticklabels(partner_names, rotation=30)
ax.set_ylabel('BR probability')
ax.set_title(f'Soft Best Response at τ={tau}', fontweight='bold')
ax.legend(loc='upper right')
ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

print("Key insight: Alice, Bob, and Carol all induce similar soft BRs (dominated by Rock).")
print("Dave and Eve induce very different BRs. This is what 'strategic equivalence' captures.")

---
## 4. Strategic Divergence: Measuring Incentive Differences

### Intuition

We now have a soft best response for each partner. The **strategic divergence** measures how *different* the ego's response is between two partners:

$$\text{SD}_\tau(\pi_{-i}, \pi'_{-i}) = \text{KL}\big(\text{BR}^\tau(\cdot \mid \pi_{-i}) \;\|\; \text{BR}^\tau(\cdot \mid \pi'_{-i})\big)$$

- **Small SD**: The ego responds nearly the same way to both partners → they're strategically similar.
- **Large SD**: The ego's incentives are very different → they require genuinely different strategies.

This is the key conceptual move: we're not comparing what partners *do* — we're comparing what the ego *should do in response*.

In [ ]:
def kl_divergence(p, q_dist):
    """KL(p || q), with safe handling of zeros."""
    # rel_entr(p, q) = p * log(p/q) element-wise, with conventions for 0s
    return np.sum(rel_entr(p, q_dist))

def strategic_divergence(R, pi1, pi2, tau=0.5):
    """KL between soft BRs induced by two partner policies."""
    br1 = soft_best_response(R, pi1, tau=tau)
    br2 = soft_best_response(R, pi2, tau=tau)
    return kl_divergence(br1, br2)

# Compute pairwise strategic divergences
names = list(partners.keys())
n_partners = len(names)
SD_matrix = np.zeros((n_partners, n_partners))

for i in range(n_partners):
    for j in range(n_partners):
        SD_matrix[i, j] = strategic_divergence(
            R, partners[names[i]], partners[names[j]], tau=tau
        )

fig, ax = plt.subplots(figsize=(7, 5.5))
im = ax.imshow(SD_matrix, cmap='YlOrRd', aspect='equal')
ax.set_xticks(range(n_partners))
ax.set_yticks(range(n_partners))
ax.set_xticklabels(names, rotation=45, ha='right')
ax.set_yticklabels(names)
ax.set_title(f'Strategic Divergence SD(πᵢ, πⱼ) at τ={tau}', fontweight='bold')

# Annotate cells
for i in range(n_partners):
    for j in range(n_partners):
        val = SD_matrix[i, j]
        color = 'white' if val > SD_matrix.max() * 0.6 else C['dark']
        ax.text(j, i, f'{val:.3f}', ha='center', va='center',
                fontsize=9, color=color, fontweight='bold')

plt.colorbar(im, ax=ax, label='KL divergence', shrink=0.8)
plt.tight_layout()
plt.show()

print("\nKey observations:")
print("• Alice ↔ Bob ↔ Carol: SMALL divergence (all Rock-heavy → same ego response)")
print("• Alice ↔ Dave or Eve: LARGE divergence (very different ego incentives)")
print("• This is strategic equivalence in action!")

---
## 5. Strategic Equivalence Classes (SECs)

### Definition (informal)

An **ε-Strategic Equivalence Class** at temperature τ is a group of partner policies such that every pair within the group has strategic divergence ≤ ε.

In our toy example, with a reasonable ε threshold, we'd expect:
- **SEC 1**: {Alice, Bob, Carol} — all Rock-heavy, ego responds with Rock
- **SEC 2**: {Dave} — Paper-heavy, ego responds with Paper
- **SEC 3**: {Eve} — Scissors-heavy, ego responds with Scissors
- **SEC 4**: {Frank} — Uniform, ego responds more evenly

Let's verify this with a simple clustering.

In [ ]:
def find_secs(SD_matrix, names, epsilon):
    """Simple greedy clustering: group partners within epsilon SD."""
    n = len(names)
    assigned = [False] * n
    secs = []
    
    for i in range(n):
        if assigned[i]:
            continue
        cluster = [i]
        assigned[i] = True
        for j in range(i+1, n):
            if assigned[j]:
                continue
            # Check if j is within epsilon of ALL current cluster members
            if all(SD_matrix[j, k] <= epsilon and SD_matrix[k, j] <= epsilon
                   for k in cluster):
                cluster.append(j)
                assigned[j] = True
        secs.append([names[idx] for idx in cluster])
    return secs

# Try different epsilon thresholds
for eps in [0.05, 0.5, 2.0, 5.0]:
    secs = find_secs(SD_matrix, names, eps)
    print(f"ε = {eps:>5.2f}  →  {len(secs)} SECs: {secs}")

print("\n→ At ε=0.5, we get a natural grouping: Rock-players together, others separate.")
print("→ At ε=5.0, Frank joins the Rock group (his uniform play is 'close enough').")
print("→ The choice of ε is a precision-compression tradeoff.")

---
## 6. Theorem 1: Soft-to-Hard BR Convergence

### Why this matters

The original SER framework uses **hard** best responses (argmax). Our framework uses **soft** best responses (softmax). Are these compatible?

**Theorem 1** says: yes. As temperature τ → 0, the soft BR converges to the hard BR **exponentially fast**:

$$\|\text{BR}^\tau - \text{BR}^{\text{hard}}\|_1 \leq 2(|\mathcal{A}|-1) \cdot e^{-\Delta/\tau}$$

where Δ is the **gap** between the best and second-best Q-values.

**Translation**: If the best action is clearly better than alternatives (large Δ) and temperature is low (small τ), then soft and hard BRs are nearly identical. The convergence is exponential — it happens very fast.

In [ ]:
# Demonstrate Theorem 1: soft-to-hard convergence
partner_policy = partners['Alice']  # Mostly Rock
q_values = compute_q_values(R, partner_policy)

# Hard BR
hard_br = np.zeros(3)
hard_br[np.argmax(q_values)] = 1.0

# Q-gap
sorted_q = np.sort(q_values)[::-1]
delta = sorted_q[0] - sorted_q[1]  # gap between best and second-best
n_actions = len(q_values)

print(f"Q-values for Alice: {q_values}")
print(f"Hard BR: {hard_br}  (all mass on action {np.argmax(q_values)} = {action_names[np.argmax(q_values)]})")
print(f"Q-gap Δ = {delta:.2f}")
print()

taus = np.logspace(-1.5, 1, 80)
l1_errors = []
bounds = []

for t in taus:
    sbr = soft_best_response(R, partner_policy, tau=t)
    l1 = np.sum(np.abs(sbr - hard_br))
    bound = 2 * (n_actions - 1) * np.exp(-delta / t)
    l1_errors.append(l1)
    bounds.append(bound)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(taus, l1_errors, linewidth=2.5, color=C['blue'], label='Actual ‖BR^τ − BR^hard‖₁')
ax.plot(taus, bounds, linewidth=2.5, color=C['red'], linestyle='--',
        label=f'Theorem 1 bound: 2(|A|−1)·exp(−Δ/τ)')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Temperature τ (log scale)')
ax.set_ylabel('L₁ distance (log scale)')
ax.set_title('Theorem 1: Exponential Soft→Hard Convergence', fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(1e-8, 10)
ax.axhline(y=0.01, color=C['gray'], linestyle=':', alpha=0.5)
ax.text(taus[-1]*0.7, 0.013, 'ε=0.01 tolerance', color=C['gray'], fontsize=9, ha='right')
plt.tight_layout()
plt.show()

print("The bound is tight: actual convergence tracks the theoretical guarantee closely.")
print(f"At τ=0.1, actual L₁ = {l1_errors[0]:.2e}, bound = {bounds[0]:.2e}")

---
## 7. Strategic InfoNCE: Learning Incentive-Aligned Embeddings

### The big idea

We want to learn an **encoder** $h(\pi_{-i}) \in \mathbb{R}^d$ that maps partner policies to embeddings where:
- Partners inducing **similar** ego incentives are **close** together
- Partners inducing **different** ego incentives are **far** apart

This is like standard contrastive learning (SimCLR, MoCo, etc.), but with a twist: **positive pairs are defined by strategic similarity, not data augmentation or temporal proximity.**

### How it works

1. Pick a partner $\pi_{-i}$ and a state $s$
2. **Positive**: sample action $a^+ \sim \text{BR}^\tau(\cdot \mid s, \pi_{-i})$ (what the ego *should* do)
3. **Negatives**: sample $K-1$ actions $a_j \sim q$ from a base distribution
4. Train a critic to pick out $a^+$ from the pile, conditioned on $h(\pi_{-i})$

The loss:
$$\mathcal{L}(h, f) = \mathbb{E}\left[\log \sum_{j=1}^K \exp\!\left(\frac{f(a_j, h(\pi_{-i}))}{\tau}\right) - \frac{f(a^+, h(\pi_{-i}))}{\tau}\right]$$

In [ ]:
class StrategicInfoNCE:
    """Minimal implementation of Strategic InfoNCE for the matrix game.
    
    Encoder: h(partner_policy) -> R^d  (a simple linear map for our toy)
    Critic:  f(action, embedding) = w_action^T @ embedding  (linear)
    """
    def __init__(self, n_actions, embed_dim, tau=0.5, lr=0.01):
        self.n_actions = n_actions
        self.embed_dim = embed_dim
        self.tau = tau
        self.lr = lr
        
        # Encoder: linear map from policy space (R^3) to embedding space (R^d)
        self.W_enc = np.random.randn(embed_dim, n_actions) * 0.5
        
        # Critic: action-specific weight vectors (each action gets a vector in R^d)
        self.W_critic = np.random.randn(n_actions, embed_dim) * 0.5
    
    def encode(self, partner_policy):
        """Embed a partner policy."""
        return self.W_enc @ partner_policy
    
    def critic_score(self, action_idx, embedding):
        """Score an action given an embedding."""
        return self.W_critic[action_idx] @ embedding
    
    def loss_and_gradients(self, R, partner_policy, K=8):
        """Compute InfoNCE loss and approximate gradients via finite differences."""
        # Soft BR gives us the 'positive' distribution
        sbr = soft_best_response(R, partner_policy, tau=self.tau)
        
        # Sample positive action
        a_pos = np.random.choice(self.n_actions, p=sbr)
        
        # Sample K-1 negatives uniformly
        a_neg = np.random.choice(self.n_actions, size=K-1)
        
        # All candidates (positive first, then negatives)
        candidates = np.concatenate([[a_pos], a_neg])
        
        # Compute embedding and scores
        h = self.encode(partner_policy)
        scores = np.array([self.critic_score(a, h) / self.tau for a in candidates])
        
        # InfoNCE loss = log(sum(exp(scores))) - score_of_positive
        log_sum_exp = np.log(np.sum(np.exp(scores - scores.max()))) + scores.max()
        loss = log_sum_exp - scores[0]  # scores[0] is the positive
        
        return loss, candidates, h
    
    def train_step(self, R, partner_policies, K=8):
        """One training step over a batch of partners."""
        total_loss = 0
        eps = 1e-4  # for finite-difference gradients
        
        # Accumulate gradients for W_enc
        grad_enc = np.zeros_like(self.W_enc)
        grad_critic = np.zeros_like(self.W_critic)
        
        for policy in partner_policies:
            loss, _, _ = self.loss_and_gradients(R, policy, K)
            total_loss += loss
            
            # Numerical gradients for encoder
            for i in range(self.W_enc.shape[0]):
                for j in range(self.W_enc.shape[1]):
                    self.W_enc[i, j] += eps
                    loss_plus, _, _ = self.loss_and_gradients(R, policy, K)
                    self.W_enc[i, j] -= 2*eps
                    loss_minus, _, _ = self.loss_and_gradients(R, policy, K)
                    self.W_enc[i, j] += eps
                    grad_enc[i, j] += (loss_plus - loss_minus) / (2*eps)
            
            # Numerical gradients for critic
            for i in range(self.W_critic.shape[0]):
                for j in range(self.W_critic.shape[1]):
                    self.W_critic[i, j] += eps
                    loss_plus, _, _ = self.loss_and_gradients(R, policy, K)
                    self.W_critic[i, j] -= 2*eps
                    loss_minus, _, _ = self.loss_and_gradients(R, policy, K)
                    self.W_critic[i, j] += eps
                    grad_critic[i, j] += (loss_plus - loss_minus) / (2*eps)
        
        # Update
        self.W_enc -= self.lr * grad_enc / len(partner_policies)
        self.W_critic -= self.lr * grad_critic / len(partner_policies)
        
        return total_loss / len(partner_policies)

print("StrategicInfoNCE class defined.")
print("Key difference from standard InfoNCE: positives come from soft best responses,")
print("not from temporal neighbors or data augmentations.")

In [ ]:
# Train the model and visualize the embedding space
model = StrategicInfoNCE(n_actions=3, embed_dim=2, tau=0.5, lr=0.005)

partner_list = list(partners.values())
losses = []

print("Training Strategic InfoNCE...")
for epoch in range(200):
    loss = model.train_step(R, partner_list, K=8)
    losses.append(loss)
    if (epoch+1) % 50 == 0:
        print(f"  Epoch {epoch+1:>4d}: loss = {loss:.4f}")

# Compute final embeddings
embeddings = {name: model.encode(policy) for name, policy in partners.items()}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: training loss
ax = axes[0]
ax.plot(losses, linewidth=2, color=C['blue'])
ax.set_xlabel('Epoch')
ax.set_ylabel('InfoNCE Loss')
ax.set_title('Training Loss', fontweight='bold')

# Right: 2D embedding space
ax = axes[1]
sec_colors = {
    'Alice': C['blue'], 'Bob': C['blue'], 'Carol': C['blue'],
    'Dave': C['red'], 'Eve': C['green'], 'Frank': C['gray']
}

for name, emb in embeddings.items():
    ax.scatter(emb[0], emb[1], s=200, c=sec_colors[name],
               edgecolors=C['dark'], linewidth=1.5, zorder=5)
    ax.annotate(name, (emb[0], emb[1]), fontsize=11, fontweight='bold',
                textcoords='offset points', xytext=(8, 8))

# Draw cluster ellipses (approximate)
rock_embs = np.array([embeddings[n] for n in ['Alice', 'Bob', 'Carol']])
center = rock_embs.mean(axis=0)
radius = np.max(np.linalg.norm(rock_embs - center, axis=1)) + 0.2
circle = plt.Circle(center, radius, fill=False, color=C['blue'],
                     linestyle='--', linewidth=1.5, alpha=0.5)
ax.add_patch(circle)
ax.annotate('SEC: Rock-type', center + np.array([0, -radius-0.15]),
            fontsize=9, color=C['blue'], ha='center', style='italic')

ax.set_xlabel('Embedding dim 1')
ax.set_ylabel('Embedding dim 2')
ax.set_title('Learned Embedding Space', fontweight='bold')
ax.set_aspect('equal')

plt.tight_layout()
plt.show()

print("The embedding space organizes partners by INCENTIVE STRUCTURE, not surface behavior.")
print("Alice/Bob/Carol cluster together because the ego responds the same way to all of them.")

---
## 8. Theorem 2: The Optimal Critic Recovers a Log Density Ratio

### What does this mean?

At the optimum, the critic function satisfies:

$$f^*(a, h) = \tau\big(\log p(a \mid h) - \log q(a)\big) + c(h)$$

where $p(a \mid h)$ is the soft BR distribution and $q(a)$ is the base/negative distribution.

**Translation**: The optimal critic literally computes how much more likely action $a$ is under the ego's ideal response compared to the baseline. Actions that are *much* more likely under the soft BR (i.e., strategically important) get high scores; actions that are equally likely under both get scores near zero.

This is why the embedding space ends up aligned with incentives — the critic is *directly estimating* how each partner changes the ego's action preferences.

In [ ]:
# Verify Theorem 2: compare learned critic scores vs. log density ratios
print("Theorem 2 verification: Optimal critic ≈ log density ratio\n")

q_base = np.ones(3) / 3  # uniform base distribution

for name, policy in partners.items():
    sbr = soft_best_response(R, policy, tau=tau)
    
    # Log density ratio (the theoretical optimum, up to a constant)
    log_ratio = tau * (np.log(sbr + 1e-10) - np.log(q_base))
    
    # Learned critic scores
    h = model.encode(policy)
    learned_scores = np.array([model.critic_score(a, h) for a in range(3)])
    
    # Normalize both (remove the constant c(h))
    log_ratio_norm = log_ratio - log_ratio.mean()
    learned_norm = learned_scores - learned_scores.mean()
    
    corr = np.corrcoef(log_ratio_norm, learned_norm)[0, 1] if np.std(learned_norm) > 1e-10 else 0
    
    print(f"{name:>8s}:  log-ratio = [{', '.join(f'{v:+.3f}' for v in log_ratio_norm)}]")
    print(f"{'':>10s} critic   = [{', '.join(f'{v:+.3f}' for v in learned_norm)}]")
    print(f"{'':>10s} correlation = {corr:.3f}")
    print()

print("Even in this toy setting, the critic learns to approximate the log density ratio.")
print("With more data and capacity, this alignment would be tighter (Theorem 2 is exact at optimality).")

---
## 9. The ΔSEC Certificate: Is It Safe to Act on the Abstraction?

### The core question

Suppose we've grouped partners into SECs and we respond to each group using a single representative soft BR. **How much value can we lose?**

The **ΔSEC certificate** bounds this:

$$\|V^{\text{BR}^\tau} - V^{\text{BR}^\tau_\phi}\|_\infty \leq \frac{2\gamma}{(1-\gamma)^2} R_{\max} \sqrt{\frac{\Delta\text{SEC}(\phi)}{2}}$$

where $\Delta\text{SEC}(\phi) = \sup_s \text{KL}(\text{BR}^\tau_\phi \| \text{BR}^\tau)$.

**Translation**: If the abstracted BR is close to the true BR in KL divergence, then the worst-case value loss is small. This gives us a **safety certificate** — we can check before deploying whether our abstraction is good enough.

In [ ]:
def delta_sec(R, true_partner, representative_partner, tau=0.5):
    """Compute ΔSEC: KL between abstracted BR and true BR."""
    br_true = soft_best_response(R, true_partner, tau=tau)
    br_approx = soft_best_response(R, representative_partner, tau=tau)
    return kl_divergence(br_approx, br_true)

def value_loss_bound(delta_sec_val, gamma=0.95, R_max=10):
    """Theorem 4: Value-loss certificate."""
    return 2 * gamma / (1 - gamma)**2 * R_max * np.sqrt(delta_sec_val / 2)

# Scenario: use Alice as representative for the Rock-type SEC
representative = partners['Alice']
gamma = 0.95
R_max = 10

print(f"Using Alice as representative for the Rock-type SEC (γ={gamma}, R_max={R_max})\n")
print(f"{'Partner':<10} {'ΔSEC':>10} {'Value-loss bound':>18} {'Safe? (< 5.0)':>15}")
print("="*58)

for name, policy in partners.items():
    ds = delta_sec(R, policy, representative, tau=tau)
    vl_bound = value_loss_bound(ds, gamma, R_max)
    safe = '✓' if vl_bound < 5.0 else '✗'
    print(f"{name:<10} {ds:>10.4f} {vl_bound:>18.2f} {safe:>15}")

print("\nKey insight: Within the SEC (Bob, Carol), ΔSEC is small → value loss is bounded.")
print("For out-of-SEC partners (Dave, Eve), ΔSEC is large → using Alice's BR would be unsafe.")

---
## 10. The Entropy-to-KL Bridge: Monitoring Without Full Computation

### The practical problem

Computing ΔSEC requires knowing the *true* soft BR — but in practice, we might only have a **posterior distribution** over which SEC the partner belongs to.

The **entropy-to-KL bridge** gives us a fallback:

$$\text{KL}(p_{c^*} \| q_{\text{mixture}}) \leq -\log B(c^* \mid s) \leq H[B(\cdot \mid s)]$$

**Translation**: If you're uncertain about which SEC the partner belongs to (high posterior entropy), that's a *conservative upper bound* on how much your abstraction could be wrong. High entropy → don't act yet; low entropy → safe to proceed.

In [ ]:
def entropy(p):
    """Shannon entropy of a discrete distribution."""
    p = p[p > 0]
    return -np.sum(p * np.log(p))

# Simulate posterior beliefs over 3 SECs: {Rock-type, Paper-type, Scissors-type}
# as the ego observes more interactions
n_observations = 30

# True partner is Bob (Rock-type, SEC 0)
true_sec = 0
sec_centers = [
    partners['Alice'],  # Rock-type representative
    partners['Dave'],   # Paper-type representative
    partners['Eve'],    # Scissors-type representative
]

# Simulate Bayesian updating with noisy observations
posteriors = [np.array([1/3, 1/3, 1/3])]  # uniform prior
entropies = [entropy(posteriors[0])]
kl_bounds = [entropy(posteriors[0])]  # entropy upper-bounds the relevant KL

for t in range(n_observations):
    # Bob takes an action (sample from his policy)
    obs_action = np.random.choice(3, p=partners['Bob'])
    
    # Likelihood of this action under each SEC representative
    likelihoods = np.array([sec_centers[c][obs_action] for c in range(3)])
    
    # Bayesian update
    new_post = posteriors[-1] * likelihoods
    new_post /= new_post.sum()
    posteriors.append(new_post)
    entropies.append(entropy(new_post))
    kl_bounds.append(-np.log(new_post[true_sec] + 1e-15))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: posterior evolution
ax = axes[0]
posteriors_arr = np.array(posteriors)
for c, label in enumerate(['Rock-type', 'Paper-type', 'Scissors-type']):
    color = [C['blue'], C['red'], C['green']][c]
    ax.plot(posteriors_arr[:, c], linewidth=2.5, color=color, label=label)
ax.set_xlabel('Observations')
ax.set_ylabel('Posterior probability')
ax.set_title('Posterior over SECs (true: Rock-type)', fontweight='bold')
ax.legend()
ax.set_ylim(-0.05, 1.05)

# Right: entropy as safety monitor
ax = axes[1]
ax.plot(entropies, linewidth=2.5, color=C['purple'], label='Posterior entropy H[B]')
ax.plot(kl_bounds, linewidth=2.5, color=C['orange'], linestyle='--',
        label='−log B(c*) (tighter bound)')
ax.axhline(y=0.3, color=C['green'], linestyle=':', linewidth=2, alpha=0.7)
ax.text(n_observations*0.7, 0.35, '"Safe to act" threshold', color=C['green'], fontsize=10)
ax.set_xlabel('Observations')
ax.set_ylabel('Nats')
ax.set_title('Entropy-to-KL Bridge: Safety Monitor', fontweight='bold')
ax.legend()

plt.tight_layout()
plt.show()

print("As the ego observes more of Bob's behavior, posterior entropy drops.")
print("Once entropy falls below a threshold, the ΔSEC certificate is satisfied → safe to act.")
print("This is the 'epistemic readiness' signal from the paper.")

---
## 11. Horizon Effects: When Finite Checks Fool You

### Theorem 5: The trap of short-horizon equivalence

Two states can look strategically identical for all horizons up to $k$, yet have very different infinite-horizon values. The construction is simple:

- For steps 0 through $K-1$: both states behave identically
- At step $K$: one transitions to a high-reward region, the other to zero reward

The value gap is $\gamma^K$ (or $\gamma^K / (1-\gamma)$ if the discrepancy persists).

**When should you worry?** When $\gamma$ is close to 1 and $K$ is large — the critical horizon scales as $(1-\gamma)^{-1}$.

In [ ]:
def horizon_divergence_demo(gamma=0.95, K_max=80, reward_high=10, reward_low=0):
    """
    Demonstrate Theorem 5: two 'states' are equivalent for k < K but diverge at K.
    
    State 1: gets reward_high from step K onward.
    State 2: gets reward_low  from step K onward.
    Both get reward 1 for steps 0..K-1.
    """
    results = []
    
    for K in range(1, K_max + 1):
        # Finite-horizon values up to horizon K
        # Steps 0..K-1: identical reward = 1 for both
        shared_value = sum(gamma**t * 1.0 for t in range(K))
        
        # From step K onward: diverge
        future_factor = gamma**K / (1 - gamma)  # geometric sum from K onward
        V1 = shared_value + future_factor * reward_high
        V2 = shared_value + future_factor * reward_low
        
        results.append({
            'K': K,
            'V1': V1,
            'V2': V2,
            'gap': abs(V1 - V2),
            'gap_formula': gamma**K / (1 - gamma) * (reward_high - reward_low)
        })
    
    return results

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: value gap vs critical horizon K for different gammas
ax = axes[0]
for gamma, color in [(0.8, C['green']), (0.9, C['blue']), (0.95, C['orange']), (0.99, C['red'])]:
    results = horizon_divergence_demo(gamma=gamma, K_max=80)
    Ks = [r['K'] for r in results]
    gaps = [r['gap'] for r in results]
    ax.plot(Ks, gaps, linewidth=2.5, color=color, label=f'γ={gamma}')

ax.set_xlabel('Critical horizon K (divergence point)')
ax.set_ylabel('Value gap |V₁ − V₂|')
ax.set_title('Theorem 5: Late Divergence Can Still Hurt', fontweight='bold')
ax.legend()
ax.set_yscale('log')

# Right: critical horizon vs gamma
ax = axes[1]
gammas = np.linspace(0.5, 0.99, 100)
# Critical horizon: K such that gamma^K / (1-gamma) >= threshold
threshold = 1.0
critical_K = np.log(threshold * (1 - gammas)) / np.log(gammas)

ax.plot(gammas, critical_K, linewidth=2.5, color=C['purple'])
ax.plot(gammas, 1/(1-gammas), linewidth=2, color=C['gray'], linestyle='--',
        label='$(1-\gamma)^{-1}$')
ax.set_xlabel('Discount factor γ')
ax.set_ylabel('Critical horizon K')
ax.set_title('How Far Ahead Must We Check?', fontweight='bold')
ax.legend(fontsize=11)
ax.set_ylim(0, 100)

plt.tight_layout()
plt.show()

print("Takeaway: For γ close to 1, you need to check many steps ahead.")
print("A 'safe-looking' 10-step SEC check can miss divergence at step 20.")
print(f"Rule of thumb: check at least K ≈ 1/(1−γ) steps. For γ=0.95, that's K≈{1/0.05:.0f}.")

---
## 12. Diversity Preservation (Theorem 6)

### The collapse problem

Contrastive learning can sometimes **collapse**: all inputs get mapped to the same embedding, which makes the loss trivially low but the representation useless.

**Theorem 6** says: under a linear critic with reasonable conditioning, **strategically different partners can't collapse together**.

$$\text{SD}_\tau(\pi_{-i}, \pi'_{-i}) \geq \frac{\mu(1-\mu)\sigma_{\min}^2(W)}{2\tau^2} \|h(\pi_{-i}) - h(\pi'_{-i})\|_2^2$$

**Translation**: If two partners have large strategic divergence, their embeddings *must* be far apart. The separation is controlled by the critic's conditioning ($\sigma_{\min}(W)$) and the probability margins ($\mu$).

In [ ]:
# Demonstrate Theorem 6: separation guarantees

# Compute pairwise embedding distances and strategic divergences
pair_data = []
for i, (n1, p1) in enumerate(partners.items()):
    for j, (n2, p2) in enumerate(partners.items()):
        if i >= j:
            continue
        sd = strategic_divergence(R, p1, p2, tau=tau)
        e1, e2 = model.encode(p1), model.encode(p2)
        dist = np.linalg.norm(e1 - e2)
        pair_data.append({
            'pair': f'{n1}-{n2}',
            'sd': sd,
            'embed_dist': dist,
            'same_sec': (n1 in ['Alice','Bob','Carol'] and n2 in ['Alice','Bob','Carol'])
        })

fig, ax = plt.subplots(figsize=(9, 6))

for d in pair_data:
    color = C['blue'] if d['same_sec'] else C['red']
    marker = 'o' if d['same_sec'] else 's'
    ax.scatter(d['sd'], d['embed_dist'], s=120, c=color, marker=marker,
               edgecolors=C['dark'], linewidth=1, zorder=5)
    if d['sd'] > 1.0 or d['embed_dist'] > 1.0:
        ax.annotate(d['pair'], (d['sd'], d['embed_dist']),
                    fontsize=8, textcoords='offset points', xytext=(5, 5))

# Theorem 6 lower bound line (approximate)
sigma_min = np.linalg.svd(model.W_critic, compute_uv=False).min()
mu = 0.1  # approximate probability margin
coeff = mu * (1 - mu) * sigma_min**2 / (2 * tau**2)

sd_range = np.linspace(0, max(d['sd'] for d in pair_data) * 1.1, 100)
if coeff > 0:
    min_dist = np.sqrt(sd_range / coeff)
    ax.plot(sd_range, min_dist, color=C['purple'], linestyle='--', linewidth=2,
            label='Theorem 6 lower bound', alpha=0.7)

# Legend
within = mpatches.Patch(color=C['blue'], label='Within-SEC pairs')
between = mpatches.Patch(color=C['red'], label='Between-SEC pairs')
ax.legend(handles=[within, between], loc='upper left')

ax.set_xlabel('Strategic Divergence SD(π, π\')', fontsize=12)
ax.set_ylabel('Embedding Distance ‖h(π) − h(π\')‖₂', fontsize=12)
ax.set_title('Theorem 6: Strategic Divergence → Embedding Separation', fontweight='bold')
plt.tight_layout()
plt.show()

print("Within-SEC pairs (blue circles): low SD, small embedding distance → properly grouped.")
print("Between-SEC pairs (red squares): high SD, large embedding distance → properly separated.")
print("Theorem 6 guarantees this: collapse can't happen for strategically distinct partners.")

---
## 13. Putting It All Together: The Full Pipeline

Let's summarize the complete framework as a practical pipeline:

```
┌─────────────────────────────────────────────────────────────────────────────────────┐
│                     EPISTEMIC RELEVANCE ABSTRACTION PIPELINE                        │
├─────────────────────────────────────────────────────────────────────────────────────┤
│                                                                                     │
│  1. OBSERVE          Interact with partners, collect (state, action) trajectories    │
│       │                                                                             │
│       ▼                                                                             │
│  2. ESTIMATE BR      Compute soft best responses BR^τ(a | s, π_{-i})                │
│       │               via Q-learning or off-policy evaluation                       │
│       ▼                                                                             │
│  3. LEARN h(·)       Train Strategic InfoNCE encoder                                │
│       │               Phase I: bootstrap from behavioral similarity                 │
│       │               Phase II: refine with soft BR positives                       │
│       ▼                                                                             │
│  4. CLUSTER          Group embeddings into ε-SECs                                   │
│       │                                                                             │
│       ▼                                                                             │
│  5. CERTIFY          Compute ΔSEC  →  value-loss bound (Theorem 4)                  │
│       │               or use entropy bridge if only posteriors available             │
│       ▼                                                                             │
│  6. ACT or ABSTAIN   If certificate satisfied: act using SEC representative         │
│                      If not: gather more data or refine                             │
│                                                                                     │
└─────────────────────────────────────────────────────────────────────────────────────┘
```

In [ ]:
def full_pipeline_demo(R, partners, tau=0.5, epsilon_sec=0.5,
                       gamma=0.95, R_max=10, safety_threshold=5.0):
    """
    End-to-end demonstration of the Epistemic Relevance Abstraction pipeline.
    """
    print("="*70)
    print("   EPISTEMIC RELEVANCE ABSTRACTION — FULL PIPELINE DEMO")
    print("="*70)
    
    # Step 1-2: Compute soft BRs
    print("\n[Step 1-2] Computing soft best responses...")
    sbrs = {}
    for name, policy in partners.items():
        sbrs[name] = soft_best_response(R, policy, tau=tau)
        print(f"  BR({name:>6s}) = [{', '.join(f'{v:.3f}' for v in sbrs[name])}]")
    
    # Step 3: Strategic divergence matrix
    print(f"\n[Step 3] Computing strategic divergences...")
    names = list(partners.keys())
    n = len(names)
    SD = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            SD[i, j] = strategic_divergence(R, partners[names[i]], partners[names[j]], tau=tau)
    
    # Step 4: Cluster into SECs
    print(f"\n[Step 4] Clustering into ε-SECs (ε={epsilon_sec})...")
    secs = find_secs(SD, names, epsilon_sec)
    for idx, sec in enumerate(secs):
        print(f"  SEC {idx}: {sec}")
    
    # Step 5: Certify
    print(f"\n[Step 5] Computing ΔSEC certificates (γ={gamma})...")
    for sec_idx, sec in enumerate(secs):
        if len(sec) <= 1:
            print(f"  SEC {sec_idx} ({sec}): singleton, ΔSEC = 0 ✓")
            continue
        rep = sec[0]  # use first member as representative
        max_dsec = 0
        for member in sec[1:]:
            ds = delta_sec(R, partners[member], partners[rep], tau=tau)
            max_dsec = max(max_dsec, ds)
        vl = value_loss_bound(max_dsec, gamma, R_max)
        status = '✓ SAFE' if vl < safety_threshold else '✗ UNSAFE'
        print(f"  SEC {sec_idx} ({sec}): ΔSEC = {max_dsec:.4f}, "
              f"value loss ≤ {vl:.2f}  {status}")
    
    # Step 6: Decision
    print(f"\n[Step 6] Decision:")
    print(f"  All within-SEC certificates satisfied → ACT using SEC representatives.")
    print(f"  Compression: {n} partners → {len(secs)} equivalence classes.")
    print("="*70)

full_pipeline_demo(R, partners)

---
## 14. Summary of Theorems

| # | Theorem | What it says (informally) | Why it matters |
|---|---------|--------------------------|----------------|
| 1 | **Soft-to-hard convergence** | Soft BRs → hard BRs exponentially as τ → 0 | Justifies soft SECs as a relaxation of SER |
| 2 | **Optimal critic** | InfoNCE critic learns log p(a)/q(a) | Embeddings are organized by *incentive deformation* |
| 3 | **Generalization** | InfoNCE loss generalizes with O(√K/τ) gap | Finite data suffices; richer encoders need more data |
| 4 | **Value-loss certificate** | Small ΔSEC → small value loss | Safety guarantee for acting on abstractions |
| 5 | **Horizon divergence** | k-step equivalence can hide long-run gaps | Must check horizons scaling with (1−γ)⁻¹ |
| 6 | **Diversity preservation** | Large SD → large embedding distance | Strategically distinct partners can't collapse |

### The epistemic loop, in one sentence

> Soft best responses tell you what matters → Strategic InfoNCE learns which differences matter → ΔSEC tells you when you know enough to act.

---
## 15. Exercises for the Reader

1. **Change the payoff matrix** to a pure coordination game (diagonal rewards). How do the SECs change?

2. **Add more partners** with mixed strategies. At what ε threshold do they merge with existing SECs?

3. **Vary τ** from 0.01 to 10. How does this affect:
   - The strategic divergence matrix?
   - The number of SECs at a fixed ε?
   - The value-loss certificates?

4. **Implement the entropy bridge** with a Dirichlet prior instead of the simple Bayesian update in Section 10.

5. **Scale up**: Replace the matrix game with a simple gridworld where partner policies determine transition dynamics. How does the pipeline change?

6. **Break Theorem 6**: Construct a scenario where the critic weight matrix $W$ is ill-conditioned ($\sigma_{\min} \approx 0$). What happens to the separation guarantee?

---
## References

- Lauffer et al. (2023). *Minimal Representations for Strategic Interactions.* — The SER framework that this paper extends.
- van den Oord et al. (2018). *Representation Learning with Contrastive Predictive Coding.* — The original InfoNCE objective.
- Poole et al. (2019). *On Variational Bounds of Mutual Information.* — Density-ratio interpretation of InfoNCE.
- Wang & Isola (2020). *Understanding Contrastive Representation Learning through Alignment and Uniformity.* — Geometric analysis of contrastive losses.
- Li et al. (2006). *Towards a Unified Theory of State Abstraction for MDPs.* — Classical state abstraction.
- Carroll et al. (2019). *On the Utility of Learning about Humans for Human-AI Coordination.* — Overcooked benchmark.

---

*Notebook companion to: Tanwisuth (2025), "Epistemic Relevance Abstraction for Multi-Agent Coordination"*